# 15分钟城市时间贫困研究 — GIS数据处理流水线

**研究区域**: 深圳市南山区  
**研究目的**: 基于POI设施与小区AOI数据，计算社区时间可达性贫困指数(CTPI)，识别服务盲区

---

## # 环境配置与依赖安装

本节安装并验证所有必需的Python包，确保中文字体正确渲染。

In [ ]:
# 安装所需依赖包
!pip install -q geopandas shapely pyproj folium matplotlib pandas numpy requests pandas fiona

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
import pandas as pd
import numpy as np
import geopandas as gpd
import warnings

warnings.filterwarnings('ignore')

# 配置中文字体支持
plt.rcParams['font.sans-serif'] = [
    'Microsoft YaHei', 'SimHei', 'Noto Sans CJK SC', 'Noto Sans SC',
    'SimSun', 'AR PL UMing CN', 'WenQuanYi Micro Hei', 'Arial Unicode MS', 'DejaVu Sans'
]
plt.rcParams['axes.unicode_minus'] = False

print(f"Python 版本: {__import__('sys').version}")
print(f"Pandas 版本: {pd.__version__}")
print(f"NumPy 版本: {np.__version__}")
print(f"GeoPandas 版本: {gpd.__version__}")
print(f"Matplotlib 版本: {matplotlib.__version__}")

---

## # 1. 数据导入与探索性分析

加载POI点数据和小区AOI面数据，检查数据基本情况。

In [ ]:
import os
from shapely.geometry import Point, Polygon, box
from pyproj import CRS
import requests
import json
import folium
from folium.plugins import HeatMap

BASE_DIR = r"e:\xicha gis 智能定位\15分钟城市时间贫困研究"
os.makedirs(BASE_DIR, exist_ok=True)
print(f"工作目录: {BASE_DIR}")

In [ ]:
# ============================================================================
# 生成模拟POI数据集（用于演示，实际使用时请替换为真实数据）
# ============================================================================

np.random.seed(42)

# 南山区中心坐标约: (113.93, 22.53)
POI_TYPES = {
    '便利店': 120,
    '药店': 45,
    '医院': 8,
    '诊所': 35,
    '快递站': 60,
    '银行': 20,
    'ATM': 80,
    '菜市场': 15,
    '学校': 12,
    '幼儿园': 25,
    '公交站': 200,
    '地铁站': 8,
    '公园': 18,
    '餐厅': 150,
    '咖啡店': 40,
    '健身房': 20,
    '游乐场': 10
}

poi_records = []
poi_id = 1

for poi_type, count in POI_TYPES.items():
    for _ in range(count):
        lng = np.random.uniform(113.85, 113.98)
        lat = np.random.uniform(22.48, 22.54)
        # 模拟部分数据使用GCJ02坐标系（会有偏移）
        coord_system = np.random.choice(['GCJ02', 'WGS84'], p=[0.3, 0.7])
        is_24h = np.random.random() < 0.15
        
        poi_records.append({
            'poi_id': poi_id,
            'name': f"{poi_type}_{poi_id}",
            'type': poi_type,
            'lng': lng,
            'lat': lat,
            'coord_system': coord_system,
            'business_period': '24h' if is_24h else np.random.choice(['08:00-22:00', '09:00-21:00', '10:00-20:00', '06:00-23:00', np.nan]),
            'address': f"南山街道{poi_id}号",
            'rating': round(np.random.uniform(3.0, 5.0), 1) if np.random.random() > 0.2 else np.nan,
            'reviews': int(np.random.exponential(50)) if np.random.random() > 0.1 else 0
        })
        poi_id += 1

poi_df = pd.DataFrame(poi_records)
print(f"生成POI数据: {len(poi_df)} 条记录")
print(f"设施类型分布:\n{poi_df['type'].value_counts()}")

In [ ]:
# ============================================================================
# 生成模拟小区AOI数据集（用于演示）
# ============================================================================

np.random.seed(123)

community_names = [
    '大冲新城花园', '华润城润府', '香山里花园', '天鹅堡', '恒裕滨城',
    '阳光棕榈园', '鼎太风华', '招商海月花园', '半岛城邦', '海境界',
    '蔚蓝海岸', '澳城花园', '阳光带海滨城', '海怡湾', '卓越维港',
    '南头城村', '大新村', '南园村', '向北村', '田厦村',
    '南山村', '荔芳村', '登良花园', '龙佳园', '金晖嘉园',
    '保障房A区', '保障房B区', '南山保障房', '前海保障房', '科技园公租房',
    '华侨城LOFT公馆', '纯水岸', '香山里二期', '天鹅湖花园', '波托菲诺',
    '塘朗雅苑', '大学城公寓', '西丽保障房', '桃源村三期', '珠光村',
    '松坪村', '朗麓家园', '高新公寓', '深圳湾一号', '华润悦府',
    '万科云城', '深铁汇里', '金地朗悦', '龙海家园', '龙瑞家园'
]

def generate_building_polygon(center_lng, center_lat, size_meters=200):
    """在给定中心点生成矩形建筑物轮廓"""
    # 1度经度≈111km*cos(lat)，1度纬度≈111km
    lat_per_m = 1 / 111000
    lng_per_m = 1 / (111000 * np.cos(np.radians(center_lat)))
    half_w = size_meters * lng_per_m * np.random.uniform(0.4, 0.8)
    half_h = size_meters * lat_per_m * np.random.uniform(0.4, 0.8)
    angle = np.random.uniform(0, 360)
    # 简单矩形，不做旋转
    coords = [
        (center_lng - half_w, center_lat - half_h),
        (center_lng + half_w, center_lat - half_h),
        (center_lng + half_w, center_lat + half_h),
        (center_lng - half_w, center_lat + half_h),
        (center_lng - half_w, center_lat - half_h)
    ]
    return Polygon(coords)

community_records = []
for i, name in enumerate(community_names):
    lng = np.random.uniform(113.85, 113.98)
    lat = np.random.uniform(22.48, 22.54)
    size = np.random.uniform(50, 500)  # 面积(m²)，后续转km²
    
    # 根据名称模式分类
    if any(kw in name for kw in ['村', '城中村', '村']):
        community_type = '城中村'
    elif any(kw in name for kw in ['保障房', '公租房', '安居房']):
        community_type = '保障房'
    elif any(kw in name for kw in ['别墅', '纯水岸', '香山里', '天鹅', '波托']):
        community_type = '别墅区'
    else:
        community_type = '商品房'
    
    community_records.append({
        'community_id': i + 1,
        'name': name,
        'type': community_type,
        'center_lng': lng,
        'center_lat': lat,
        'polygon': generate_building_polygon(lng, lat),
        'area_m2': size,
        'floor_count': int(np.random.randint(6, 50)),
        'built_year': int(np.random.randint(2000, 2024)),
        'population': int(np.random.uniform(500, 5000))
    })

# 创建GeoDataFrame
aoi_gdf = gpd.GeoDataFrame(community_records, crs='EPSG:4326')
print(f"生成小区AOI数据: {len(aoi_gdf)} 条记录")
print(f"小区类型分布:\n{aoi_gdf['type'].value_counts()}")

In [ ]:
# 基本统计信息
print("=" * 60)
print("【POI数据基本信息】")
print("=" * 60)
print(f"总记录数: {len(poi_df)}")
print(f"列名: {list(poi_df.columns)}")
print(f"\n缺失值统计:\n{poi_df.isnull().sum()}")
print(f"\n坐标范围:")
print(f"  经度: [{poi_df['lng'].min():.6f}, {poi_df['lng'].max():.6f}]")
print(f"  纬度: [{poi_df['lat'].min():.6f}, {poi_df['lat'].max():.6f}]")

print("=" * 60)
print("【AOI数据基本信息】")
print("=" * 60)
print(f"总记录数: {len(aoi_gdf)}")
print(f"列名: {list(aoi_gdf.columns)}")
print(f"坐标系: {aoi_gdf.crs}")
print(f"\n面积统计(m²):")
print(aoi_gdf['area_m2'].describe())

In [ ]:
# 预览数据样本
print("POI数据样本(前5行):")
display(poi_df.head())

print("\nAOI数据样本(前5行):")
display(aoi_gdf.head())

---

## # 2. 坐标系统一处理

**重要说明**: 

- 中国大陆在线地图(高德、腾讯、百度)使用 **GCJ-02** 坐标系(火星坐标系)
- 国际标准为 **WGS-84** (GPS设备使用)
- 两者之间存在50-500米的系统性偏移，转换公式不公开，需使用近似算法
- 本项目统一将所有数据转换至 **WGS-84 (EPSG:4326)**

**注意**: GCJ-02 → WGS-84 的精确转换涉及复杂的非线性偏移模型。本示例使用近似线性估计，实际项目建议使用 `coord converter` 等专业库。

In [ ]:
# 模拟GCJ-02到WGS-84的坐标转换（近似算法，实际使用coord_converter库）

def gcj02_to_wgs84(lng, lat):
    """
    GCJ-02 坐标系转 WGS-84 的近似转换
    
    参数:
        lng, lat: GCJ-02 经纬度
    返回:
        (wgs_lng, wgs_lat): WGS-84 经纬度
    """
    # 深圳地区近似偏移量（实际偏移随位置变化）
    # 深圳市中心偏移量约: dLng ≈ -0.003, dLat ≈ +0.002
    a = 6378245.0  # 克拉索夫斯基椭球长半轴
    ee = 0.00669342162296594323  # 扁率
    
    def transform(lng, lat):
        d_lng = -0.0025
        d_lat = 0.0015
        return lng + d_lng, lat + d_lat
    
    return transform(lng, lat)

def detect_and_transform_crs(df, lng_col='lng', lat_col='lat', 
                             coord_col='coord_system',
                             target_crs='EPSG:4326'):
    """
    检测并转换坐标系，将所有坐标统一到目标CRS
    
    参数:
        df: POI数据DataFrame
        lng_col: 经度列名
        lat_col: 纬度列名
        coord_col: 坐标系标识列名
        target_crs: 目标坐标系(默认WGS84)
    
    返回:
        df: 转换后的DataFrame
        report: 转换报告
    """
    df = df.copy()
    
    report = {
        'original_count': len(df),
        'gcj02_count': 0,
        'wgs84_count': 0,
        'unknown_count': 0,
        'transformed_count': 0
    }
    
    if coord_col in df.columns:
        # 有坐标系标识列，根据标识转换
        for idx, row in df.iterrows():
            coord_sys = str(row.get(coord_col, 'WGS84')).upper()
            if 'GCJ' in coord_sys or 'MOBI' in coord_sys:
                wgs_lng, wgs_lat = gcj02_to_wgs84(row[lng_col], row[lat_col])
                df.at[idx, lng_col] = wgs_lng
                df.at[idx, lat_col] = wgs_lat
                df.at[idx, coord_col] = 'WGS84(已转换)'
                report['transformed_count'] += 1
                report['gcj02_count'] += 1
            else:
                report['wgs84_count'] += 1
    else:
        # 无标识列，默认视为WGS84
        report['wgs84_count'] = len(df)
        report['unknown_count'] = len(df)
    
    return df, report


# 执行坐标系转换
poi_df_transformed, crs_report = detect_and_transform_crs(poi_df)

print("=" * 60)
print("【坐标系转换报告】")
print("=" * 60)
print(f"总记录数: {crs_report['original_count']}")
print(f"GCJ-02 坐标系: {crs_report['gcj02_count']} 条 (已转换)")
print(f"WGS-84 坐标系: {crs_report['wgs84_count']} 条")
print(f"未知坐标系: {crs_report['unknown_count']} 条")
print(f"共转换: {crs_report['transformed_count']} 条记录")
print(f"\n转换后坐标范围:")
print(f"  经度: [{poi_df_transformed['lng'].min():.6f}, {poi_df_transformed['lng'].max():.6f}]")
print(f"  纬度: [{poi_df_transformed['lat'].min():.6f}, {poi_df_transformed['lat'].max():.6f}]")

In [ ]:
# 可视化坐标系转换前后对比
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
ax1.scatter(poi_df[poi_df['coord_system']=='GCJ02']['lng'], 
           poi_df[poi_df['coord_system']=='GCJ02']['lat'], 
           c='red', alpha=0.5, s=20, label='GCJ-02 (转换前)')
ax1.scatter(poi_df[poi_df['coord_system']=='WGS84']['lng'], 
           poi_df[poi_df['coord_system']=='WGS84']['lat'], 
           c='blue', alpha=0.3, s=20, label='WGS-84')
ax1.set_title('坐标系转换前')
ax1.set_xlabel('经度')
ax1.set_ylabel('纬度')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2 = axes[1]
gcj_converted = poi_df_transformed[poi_df_transformed['coord_system'].str.contains('已转换', na=False)]
wgs_unchanged = poi_df_transformed[~poi_df_transformed['coord_system'].str.contains('已转换', na=False)]
ax2.scatter(wgs_unchanged['lng'], wgs_unchanged['lat'], 
           c='blue', alpha=0.3, s=20, label='WGS-84')
ax2.scatter(gcj_converted['lng'], gcj_converted['lat'], 
           c='green', alpha=0.7, s=30, label='GCJ→WGS(已转换)', marker='x')
ax2.set_title('坐标系转换后 (统一为WGS-84)')
ax2.set_xlabel('经度')
ax2.set_ylabel('纬度')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, '坐标系转换对比.png'), dpi=150, bbox_inches='tight')
plt.show()
print("图表已保存: 坐标系转换对比.png")

---

## # 3. 研究区范围裁剪

定义南山区研究范围，使用 bounding box 进行空间裁剪。

In [ ]:
# 定义研究区范围（南山区 bounding box）
STUDY_AREA = {
    'name': '南山区',
    'lng_min': 113.85,
    'lng_max': 113.98,
    'lat_min': 22.48,
    'lat_max': 22.54
}

def create_study_area_bbox(study_area):
    """创建研究区边界框几何对象"""
    return box(
        study_area['lng_min'],
        study_area['lat_min'],
        study_area['lng_max'],
        study_area['lat_max']
    )

study_bbox = create_study_area_bbox(STUDY_AREA)
print(f"研究区: {STUDY_AREA['name']}")
print(f"经度范围: [{STUDY_AREA['lng_min']}, {STUDY_AREA['lng_max']}]")
print(f"纬度范围: [{STUDY_AREA['lat_min']}, {STUDY_AREA['lat_max']}]")
print(f"面积约: {(STUDY_AREA['lng_max']-STUDY_AREA['lng_min'])*111000*np.cos(np.radians(22.5)):.1f}km × {(STUDY_AREA['lat_max']-STUDY_AREA['lat_min'])*111000:.1f}km ≈ 约185km²")

In [ ]:
# 裁剪POI数据
def clip_poi_to_bbox(df, bbox, lng_col='lng', lat_col='lat'):
    """
    将POI数据裁剪到指定边界框内
    
    参数:
        df: POI数据
        bbox: shapely.geometry.box 边界框
        lng_col, lat_col: 经纬度列名
    
    返回:
        裁剪后的DataFrame
    """
    mask = df.apply(
        lambda row: bbox.contains(Point(row[lng_col], row[lat_col])),
        axis=1
    )
    return df[mask].copy()

print(f"POI裁剪前: {len(poi_df_transformed)} 条")
poi_clipped = clip_poi_to_bbox(poi_df_transformed, study_bbox)
print(f"POI裁剪后: {len(poi_clipped)} 条")
print(f"移除超出范围: {len(poi_df_transformed) - len(poi_clipped)} 条")

# 裁剪AOI数据
print(f"\nAOI裁剪前: {len(aoi_gdf)} 条")
aoi_clipped = aoi_gdf[aoi_gdf.geometry.intersects(study_bbox)].copy()
print(f"AOI裁剪后: {len(aoi_clipped)} 条")
print(f"移除超出范围: {len(aoi_gdf) - len(aoi_clipped)} 条")

In [ ]:
# 可视化裁剪结果
fig, ax = plt.subplots(figsize=(10, 8))

# 绘制研究区边界
study_gdf = gpd.GeoDataFrame([{'geometry': study_bbox}], crs='EPSG:4326')
study_gdf.boundary.plot(ax=ax, color='black', linewidth=2, label='研究区边界')

# 绘制POI点
for poi_type in poi_clipped['type'].unique():
    subset = poi_clipped[poi_clipped['type'] == poi_type]
    ax.scatter(subset['lng'], subset['lat'], alpha=0.6, s=15, label=f'{poi_type}({len(subset)})')

# 绘制AOI面
aoi_clipped.plot(ax=ax, facecolor='none', edgecolor='blue', linewidth=0.8, alpha=0.7)

ax.set_title(f'南山区研究区 POI与AOI分布 (POI:{len(poi_clipped)}条, AOI:{len(aoi_clipped)}条)', fontsize=14)
ax.set_xlabel('经度')
ax.set_ylabel('纬度')
ax.legend(loc='upper left', fontsize=7, ncol=2, markerscale=0.8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, '研究区裁剪结果.png'), dpi=150, bbox_inches='tight')
plt.show()
print("图表已保存: 研究区裁剪结果.png")

---

## # 4. POI数据标准化与清洗

将原始POI类型映射为统一的标准分类，并执行多步骤数据清洗流程。

In [ ]:
# ============================================================================
# POI类型标准化映射
# ============================================================================

# 定义标准POI分类及其原始类型映射
POI_TYPE_MAPPING = {
    # 便利店/小卖部
    '便利店': 'convenience_store',
    '小型超市': 'convenience_store',
    '小卖部': 'convenience_store',
    '超市': 'convenience_store',
    # 药店
    '药店': 'pharmacy',
    '药房': 'pharmacy',
    '医药': 'pharmacy',
    # 医院
    '医院': 'hospital',
    '综合医院': 'hospital',
    '三甲医院': 'hospital',
    # 诊所
    '诊所': 'clinic',
    '门诊': 'clinic',
    '卫生站': 'clinic',
    # 快递站
    '快递站': 'express_station',
    '菜鸟驿站': 'express_station',
    '快递': 'express_station',
    # 银行
    '银行': 'bank',
    '商业银行': 'bank',
    # ATM
    'ATM': 'atm',
    '自助银行': 'atm',
    '自助取款': 'atm',
    # 菜市场
    '菜市场': 'market',
    '农贸市场': 'market',
    '生鲜': 'market',
    # 学校
    '学校': 'school',
    '中学': 'school',
    '小学': 'school',
    '教育': 'school',
    # 幼儿园
    '幼儿园': 'kindergarten',
    '托儿所': 'kindergarten',
    # 公交站
    '公交站': 'bus_station',
    '公交': 'bus_station',
    '巴士站': 'bus_station',
    # 地铁站
    '地铁站': 'subway',
    '地铁': 'subway',
    '轨道交通': 'subway',
    # 公园
    '公园': 'park',
    '广场': 'park',
    '绿地': 'park',
    # 餐厅
    '餐厅': 'restaurant',
    '餐馆': 'restaurant',
    '饭店': 'restaurant',
    '美食': 'restaurant',
    # 咖啡店
    '咖啡店': 'cafe',
    '咖啡': 'cafe',
    '奶茶': 'cafe',
    # 健身房
    '健身房': 'gym',
    '健身': 'gym',
    '体育': 'gym',
    # 游乐场
    '游乐场': 'playground',
    '儿童乐园': 'playground',
    '运动场': 'playground'
}

# 标准分类中文名称映射
CATEGORY_NAMES_CN = {
    'convenience_store': '便利店',
    'pharmacy': '药店',
    'hospital': '医院',
    'clinic': '诊所',
    'express_station': '快递站',
    'bank': '银行',
    'atm': 'ATM',
    'market': '菜市场',
    'school': '学校',
    'kindergarten': '幼儿园',
    'bus_station': '公交站',
    'subway': '地铁站',
    'park': '公园',
    'restaurant': '餐厅',
    'cafe': '咖啡茶饮',
    'gym': '健身房',
    'playground': '游乐场'
}

def standardize_poi_types(df, type_col='type', mapping=POI_TYPE_MAPPING):
    """
    将原始POI类型映射为标准分类
    
    参数:
        df: POI数据
        type_col: 原始类型列名
        mapping: 类型映射字典
    
    返回:
        添加了standard_type列的DataFrame
    """
    df = df.copy()
    df['standard_type'] = df[type_col].map(mapping).fillna('other')
    unmapped = df[df['standard_type'] == 'other']['type'].unique()
    if len(unmapped) > 0:
        print(f"未映射类型: {unmapped}")
    print(f"标准分类统计:\n{df['standard_type'].value_counts()}")
    return df

poi_cleaned = standardize_poi_types(poi_clipped)

In [ ]:
def clean_business_hours(df, hours_field='business_period'):
    """
    清洗营业时间字段
    
    - 统一格式
    - 标记24小时营业
    - 标记歇业/无数据
    """
    df = df.copy()
    df['is_24h'] = df[hours_field].apply(
        lambda x: True if str(x).strip() in ['24h', '24小时', '24H', '全天'] else False
    )
    df['is_closed'] = df[hours_field].apply(
        lambda x: True if pd.isna(x) or str(x).strip() in ['歇业', '暂停营业', '关闭', ''] else False
    )
    df['business_hours_clean'] = df[hours_field].fillna('未知').replace('', '未知')
    return df

def detect_24h_pois(df):
    """
    特别检测24小时营业设施（如便利店、ATM等）
    """
    df = df.copy()
    # 24h默认类型
    default_24h_types = ['便利店', 'ATM', '药店', '医院', '诊所']
    mask_24h = df['type'].isin(default_24h_types) & (df['is_24h'] == False)
    # 部分标注为夜间营业
    df['likely_24h'] = mask_24h
    print(f"检测到可能24h设施(未标注): {mask_24h.sum()} 条")
    print(f"实际标注24h设施: {df['is_24h'].sum()} 条")
    return df

poi_cleaned = clean_business_hours(poi_cleaned)
poi_cleaned = detect_24h_pois(poi_cleaned)

In [ ]:
def remove_duplicates_by_name_location(df, tolerance_meters=30, 
                                       name_col='name', lng_col='lng', lat_col='lat'):
    """
    基于名称和位置去重
    
    逻辑:
    1. 同名设施且距离<tolerance_meters视为重复
    2. 保留评分最高/评论数最多的记录
    """
    from math import radians, cos, sin, asin, sqrt
    
    def haversine_meters(lng1, lat1, lng2, lat2):
        """计算两点间距离(米)"""
        lng1, lat1, lng2, lat2 = map(radians, [lng1, lat1, lng2, lat2])
        dlat = lat2 - lat1
        dlng = lng2 - lng1
        a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlng/2)**2
        return 2 * 6371000 * asin(sqrt(a))
    
    df = df.sort_values('reviews', ascending=False).reset_index(drop=True)
    keep_idx = []
    
    for idx, row in df.iterrows():
        is_dup = False
        for keep_i in keep_idx:
            keep_row = df.loc[keep_i]
            dist = haversine_meters(row[lng_col], row[lat_col],
                                   keep_row[lng_col], keep_row[lat_col])
            if dist < tolerance_meters and row[name_col] == keep_row[name_col]:
                is_dup = True
                break
        if not is_dup:
            keep_idx.append(idx)
    
    removed = len(df) - len(keep_idx)
    print(f"去重: 原始{len(df)}条 → 去重后{len(keep_idx)}条 (移除{removed}条重复)")
    return df.loc[keep_idx].reset_index(drop=True)

poi_cleaned = remove_duplicates_by_name_location(poi_cleaned)

In [ ]:
def validate_coordinates(df, bbox, lng_col='lng', lat_col='lat'):
    """
    验证坐标是否在合理范围内
    
    检查:
    - 不超出边界框
    - 经纬度不互换(中国: lat应在22-24之间, lng在113-120之间)
    - 不为NULL
    """
    df = df.copy()
    
    # 基本范围检查
    mask_valid = (
        (df[lng_col] >= bbox.bounds[0]) & (df[lng_col] <= bbox.bounds[2]) &
        (df[lat_col] >= bbox.bounds[1]) & (df[lat_col] <= bbox.bounds[3]) &
        (df[lng_col] > 0) & (df[lat_col] > 0)
    )
    
    # 中国坐标合理性检查
    mask_china = (df[lng_col] >= 73) & (df[lng_col] <= 135) & \
                 (df[lat_col] >= 18) & (df[lat_col] <= 54)
    
    df['coord_valid'] = mask_valid & mask_china
    
    invalid_count = (~df['coord_valid']).sum()
    print(f"坐标验证: {len(df)}条中 {invalid_count}条 坐标异常")
    if invalid_count > 0:
        print(f"异常记录:\n{df[~df['coord_valid']][[lng_col, lat_col]].head()}")
    
    return df


def flag_outliers(df, lng_col='lng', lat_col='lat', std_multiplier=3):
    """
    基于统计方法识别空间离群点
    
    使用经纬度分别的Z-score方法
    """
    df = df.copy()
    
    lng_z = np.abs((df[lng_col] - df[lng_col].mean()) / df[lng_col].std())
    lat_z = np.abs((df[lat_col] - df[lat_col].mean()) / df[lat_col].std())
    
    df['is_outlier'] = (lng_z > std_multiplier) | (lat_z > std_multiplier)
    df['lng_zscore'] = lng_z
    df['lat_zscore'] = lat_z
    
    outlier_count = df['is_outlier'].sum()
    print(f"离群点检测: {outlier_count}个离群点 (Z>{std_multiplier})")
    
    return df

poi_cleaned = validate_coordinates(poi_cleaned, study_bbox)
poi_cleaned = flag_outliers(poi_cleaned)

In [ ]:
# 执行完整清洗流水线
print("=" * 60)
print("【POI数据清洗流水线】")
print("=" * 60)
print(f"\n步骤1: 类型标准化 → {poi_cleaned['standard_type'].nunique()} 个标准分类")
print(f"步骤2: 营业时间清洗 → {poi_cleaned['is_24h'].sum()}个24h设施, {poi_cleaned['is_closed'].sum()}个歇业")
print(f"步骤3: 坐标验证 → {poi_cleaned['coord_valid'].sum()}条有效, {(~poi_cleaned['coord_valid']).sum()}条无效")
print(f"步骤4: 离群点检测 → {poi_cleaned['is_outlier'].sum()}个离群点")
print(f"\n清洗后数据量: {len(poi_cleaned)} 条")

print("\n清洗后标准分类分布:")
display(poi_cleaned['standard_type'].value_counts())

---

## # 5. 小区AOI数据处理

计算小区质心、分类社区类型、验证几何有效性。

In [ ]:
# 质心计算
aoi_clipped = aoi_clipped.copy()
aoi_clipped['centroid'] = aoi_clipped.geometry.centroid
aoi_clipped['centroid_lng'] = aoi_clipped.centroid.x
aoi_clipped['centroid_lat'] = aoi_clipped.centroid.y

# 面积计算（转换为km²）
aoi_clipped['area_km2'] = aoi_clipped.geometry.area * 111000**2 * np.cos(np.radians(22.5)) * 1e-6

# 验证几何有效性
aoi_clipped['is_valid'] = aoi_clipped.geometry.is_valid
aoi_clipped['is_simple'] = aoi_clipped.geometry.is_simple

invalid_geoms = aoi_clipped[~aoi_clipped['is_valid']]
if len(invalid_geoms) > 0:
    print(f"发现 {len(invalid_geoms)} 个无效几何，自动修复中...")
    aoi_clipped.loc[~aoi_clipped['is_valid'], 'geometry'] = \
        aoi_clipped.loc[~aoi_clipped['is_valid'], 'geometry'].buffer(0)
    aoi_clipped['is_valid'] = aoi_clipped.geometry.is_valid
    print(f"修复后: {aoi_clipped['is_valid'].sum()} / {len(aoi_clipped)} 有效")
else:
    print(f"所有 {len(aoi_clipped)} 个几何对象均有效")

# 统计信息
print(f"\n小区面积统计(km²):")
print(aoi_clipped['area_km2'].describe())

print(f"\n按类型分类的小区数量:")
print(aoi_clipped.groupby('type').agg({
    'community_id': 'count',
    'area_km2': ['mean', 'std'],
    'population': ['sum', 'mean']
}).round(3))

In [ ]:
# 社区类型分类函数（基于面积和名称的综合分类）
def classify_community_types(df, area_col='area_km2', name_col='name', type_col='type'):
    """
    综合分类社区类型
    
    分类规则:
    - 城中村: 名称含村/城中村的
    - 保障房: 名称含保障房/公租房/安居房的
    - 别墅区: 面积>0.5km² 且名称含别墅/山庄/豪宅等
    - 商品房: 其他
    """
    df = df.copy()
    
    def classify(row):
        name = str(row[name_col])
        area = row[area_col]
        
        if '村' in name:
            return '城中村'
        elif any(kw in name for kw in ['保障房', '公租房', '安居房']):
            return '保障房'
        elif area > 0.3 and any(kw in name for kw in ['别墅', '山庄', '纯水岸', '天鹅']):
            return '别墅区'
        else:
            return '商品房'
    
    df['community_type'] = df.apply(classify, axis=1)
    return df

aoi_processed = classify_community_types(aoi_clipped)

print("社区类型重新分类结果:")
print(aoi_processed['community_type'].value_counts())

# 各类型详细统计
print("\n各类型小区统计:")
display(aoi_processed.groupby('community_type').agg({
    'community_id': 'count',
    'area_km2': ['mean', 'min', 'max'],
    'population': ['sum', 'mean'],
    'floor_count': 'mean'
}).round(2))

---

## # 6. 空间关联分析

基于500米步行可达范围，计算每个小区周边各类设施的空间关联指标。

In [ ]:
# 创建POI GeoDataFrame
poi_gdf = gpd.GeoDataFrame(
    poi_cleaned.copy(),
    geometry=gpd.points_from_xy(poi_cleaned['lng'], poi_cleaned['lat']),
    crs='EPSG:4326'
)

# 步行可达半径设置（米）
WALK_BUFFER_METERS = 500

# 创建米为单位的投影坐标系用于缓冲区分析
projected_crs = 'EPSG:32650'  # UTM zone 50N (深圳地区适用)

def create_walking_buffer(gdf, radius_meters, target_crs=projected_crs):
    """
    为GeoDataFrame创建步行缓冲区
    
    参数:
        gdf: GeoDataFrame (EPSG:4326)
        radius_meters: 缓冲区半径(米)
        target_crs: 投影坐标系
    
    返回:
        投影坐标系下的缓冲区GeoDataFrame
    """
    gdf_proj = gdf.to_crs(target_crs)
    gdf_proj['buffer_geometry'] = gdf_proj.geometry.buffer(radius_meters)
    gdf_proj = gdf_proj.set_geometry('buffer_geometry')
    return gdf_proj

# 为每种设施类型创建缓冲区
facility_buffers = {}
for poi_type in poi_gdf['standard_type'].unique():
    type_pois = poi_gdf[poi_gdf['standard_type'] == poi_type]
    if len(type_pois) > 0:
        buffer_gdf = create_walking_buffer(type_pois, WALK_BUFFER_METERS)
        facility_buffers[poi_type] = buffer_gdf

print(f"已为 {len(facility_buffers)} 种设施类型创建 {WALK_BUFFER_METERS}m 步行缓冲区")
print("设施类型:", list(facility_buffers.keys()))

In [ ]:
# 空间连接: 统计每个小区周边各类设施数量
def spatial_join_facilities(aoi_gdf, facility_buffers, target_crs=projected_crs):
    """
    对每个小区执行空间连接，统计周边设施
    
    参数:
        aoi_gdf: 小区GeoDataFrame
        facility_buffers: {设施类型: 缓冲区GeoDataFrame} 字典
    
    返回:
        添加了设施统计列的小区GeoDataFrame
    """
    aoi_proj = aoi_gdf.to_crs(target_crs).copy()
    
    # 初始化设施计数列
    facility_types = list(facility_buffers.keys())
    for ft in facility_types:
        aoi_proj[f'count_{ft}'] = 0
    
    # 逐类型执行空间连接
    for ft, buf_gdf in facility_buffers.items():
        buf_proj = buf_gdf.to_crs(target_crs)
        for idx, community in aoi_proj.iterrows():
            count = buf_proj.intersects(community.geometry).sum()
            aoi_proj.at[idx, f'count_{ft}'] = count
    
    return aoi_proj

aoi_spatial = spatial_join_facilities(aoi_processed, facility_buffers)

# 显示统计结果
count_cols = [c for c in aoi_spatial.columns if c.startswith('count_')]
print("各小区设施覆盖统计(前5个小区):")
display(aoi_spatial[['name', 'community_type'] + count_cols].head())

In [ ]:
# 最近设施分析
def find_nearest_facility(community_centroid, poi_gdf, facility_type, target_crs=projected_crs):
    """
    计算从社区质心到最近设施的距离
    
    参数:
        community_centroid: shapely Point对象(质心)
        poi_gdf: POI GeoDataFrame
        facility_type: 设施类型
        target_crs: 投影坐标系
    
    返回:
        最短距离(米)
    """
    type_pois = poi_gdf[poi_gdf['standard_type'] == facility_type]
    if len(type_pois) == 0:
        return np.inf
    
    type_proj = type_pois.to_crs(target_crs)
    centroid_proj = gpd.GeoSeries([community_centroid], crs='EPSG:4326').to_crs(target_crs)[0]
    distances = type_proj.geometry.distance(centroid_proj)
    return distances.min()

# 为每个小区计算各类设施最近距离
print("计算最近设施距离（可能需要一些时间）...")
for ft in facility_buffers.keys():
    col_name = f'dist_{ft}'
    aoi_spatial[col_name] = aoi_spatial.centroid.geometry.apply(
        lambda p: find_nearest_facility(p, poi_gdf, ft)
    )
print("完成！")

# 显示最近设施距离统计
dist_cols = [c for c in aoi_spatial.columns if c.startswith('dist_')]
print("\n最近设施距离统计(米) — 各设施类型:")
dist_stats = aoi_spatial[dist_cols].describe().T
dist_stats.columns = ['计数', '均值', '标准差', '最小', '25%', '50%', '75%', '最大']
display(dist_stats.round(1))

---

## # 7. 时间可达性指数计算 (CTPI)

**CTPI (Community Time Poverty Index)** 定义:

$$CTPI = \sum_{t} \sum_{f} w_{t,f} \times \mathbb{1}(d_{f} > T_{f})$$

其中:
- $t$: 时段 (白天/晚间/深夜)
- $f$: 设施类型
- $w_{t,f}$: 时段$t$中设施$f$的权重
- $d_{f}$: 到最近$f$类设施的距离
- $T_{f}$: 设施$f$的可达阈值(米)

**CTPI越大 = 时间贫困越严重 = 设施可达性越差**

In [ ]:
# 时段定义
TIME_PERIODS = {
    'day': {
        'name': '白天',
        'hours': (6, 18),
        'description': '06:00-18:00'
    },
    'evening': {
        'name': '晚间',
        'hours': (18, 23),
        'description': '18:00-23:00'
    },
    'night': {
        'name': '深夜',
        'hours': (23, 6),
        'description': '23:00-06:00'
    }
}

# 设施权重定义（时段×设施类型矩阵）
# 权重越高表示该时段该设施越重要/稀缺
FACILITY_WEIGHTS = {
    # (时段, 设施类型): 权重
    # 白天时段
    ('day', 'convenience_store'): 1.0,
    ('day', 'pharmacy'): 1.5,
    ('day', 'hospital'): 2.0,
    ('day', 'clinic'): 1.5,
    ('day', 'express_station'): 1.0,
    ('day', 'bank'): 0.5,
    ('day', 'atm'): 0.5,
    ('day', 'market'): 1.5,
    ('day', 'school'): 2.0,
    ('day', 'kindergarten'): 2.0,
    ('day', 'bus_station'): 1.0,
    ('day', 'subway'): 1.0,
    ('day', 'park'): 0.5,
    ('day', 'restaurant'): 1.0,
    ('day', 'cafe'): 0.3,
    ('day', 'gym'): 0.5,
    ('day', 'playground'): 1.0,
    
    # 晚间时段
    ('evening', 'convenience_store'): 1.5,
    ('evening', 'pharmacy'): 2.0,
    ('evening', 'hospital'): 1.5,
    ('evening', 'clinic'): 1.5,
    ('evening', 'express_station'): 2.0,
    ('evening', 'bank'): 0.0,
    ('evening', 'atm'): 1.0,
    ('evening', 'market'): 1.0,
    ('evening', 'school'): 0.0,
    ('evening', 'kindergarten'): 0.0,
    ('evening', 'bus_station'): 1.0,
    ('evening', 'subway'): 1.0,
    ('evening', 'park'): 1.0,
    ('evening', 'restaurant'): 2.0,
    ('evening', 'cafe'): 1.0,
    ('evening', 'gym'): 1.5,
    ('evening', 'playground'): 0.5,
    
    # 深夜时段
    ('night', 'convenience_store'): 2.0,
    ('night', 'pharmacy'): 2.5,
    ('night', 'hospital'): 1.5,
    ('night', 'clinic'): 2.0,
    ('night', 'express_station'): 0.0,
    ('night', 'bank'): 0.0,
    ('night', 'atm'): 1.0,
    ('night', 'market'): 0.0,
    ('night', 'school'): 0.0,
    ('night', 'kindergarten'): 0.0,
    ('night', 'bus_station'): 0.5,
    ('night', 'subway'): 0.5,
    ('night', 'park'): 0.0,
    ('night', 'restaurant'): 1.0,
    ('night', 'cafe'): 0.5,
    ('night', 'gym'): 0.0,
    ('night', 'playground'): 0.0
}

# 可达性阈值（米）- 超过此距离视为"不可达"
ACCESSIBILITY_THRESHOLDS = {
    'convenience_store': 300,
    'pharmacy': 500,
    'hospital': 1000,
    'clinic': 500,
    'express_station': 500,
    'bank': 800,
    'atm': 300,
    'market': 800,
    'school': 1000,
    'kindergarten': 500,
    'bus_station': 300,
    'subway': 800,
    'park': 500,
    'restaurant': 500,
    'cafe': 800,
    'gym': 1000,
    'playground': 800
}

print("时段定义:")
for k, v in TIME_PERIODS.items():
    print(f"  {v['name']}: {v['description']}")

print(f"\n定义了 {len(FACILITY_WEIGHTS)} 个时段×设施权重组合")
print(f"定义了 {len(ACCESSIBILITY_THRESHOLDS)} 个可达性阈值")

In [ ]:
def calculate_ctpi(df, facility_weights, thresholds, dist_cols_prefix='dist_'):
    """
    计算社区时间可达性贫困指数 (CTPI)
    
    算法:
    对于每个时段t和每个设施f:
    - 如果 dist_{f} > threshold_{f}: 该设施不可达，加权重w_{t,f}
    - 如果 dist_{f} <= threshold_{f}: 可达，不加权重
    
    CTPI = Σ w_{t,f} (对所有不可达组合求和)
    
    参数:
        df: 小区GeoDataFrame
        facility_weights: {(时段, 设施): 权重} 字典
        thresholds: {设施类型: 阈值(米)} 字典
        dist_cols_prefix: 距离列前缀
    
    返回:
        添加了CTPI相关列的DataFrame
    """
    df = df.copy()
    
    # 初始化
    df['CTPI_total'] = 0.0
    
    for period in ['day', 'evening', 'night']:
        df[f'CTPI_{period}'] = 0.0
    
    # 获取所有设施类型
    facility_types = [c.replace(dist_cols_prefix, '') for c in df.columns 
                     if c.startswith(dist_cols_prefix)]
    
    # 计算
    for idx, row in df.iterrows():
        ctpi_total = 0.0
        ctpi_by_period = {}
        
        for ft in facility_types:
            dist_col = f'{dist_cols_prefix}{ft}'
            dist = row.get(dist_col, np.inf)
            threshold = thresholds.get(ft, 500)
            
            if dist > threshold:  # 不可达
                for period in ['day', 'evening', 'night']:
                    weight = facility_weights.get((period, ft), 0)
                    ctpi_total += weight
                    ctpi_by_period[period] = ctpi_by_period.get(period, 0) + weight
        
        df.at[idx, 'CTPI_total'] = ctpi_total
        for period, score in ctpi_by_period.items():
            df.at[idx, f'CTPI_{period}'] = score
    
    # CTPI标准化 (0-100分制)
    max_ctpi = df['CTPI_total'].max()
    if max_ctpi > 0:
        df['CTPI_normalized'] = (df['CTPI_total'] / max_ctpi) * 100
    else:
        df['CTPI_normalized'] = 0
    
    return df


aoi_ctpi = calculate_ctpi(aoi_spatial, FACILITY_WEIGHTS, ACCESSIBILITY_THRESHOLDS)

# 显示CTPI统计
print("=" * 60)
print("【CTPI计算结果】")
print("=" * 60)
print(f"\n总体CTPI统计:")
print(aoi_ctpi['CTPI_total'].describe())

print(f"\n各时段CTPI统计:")
display(aoi_ctpi[['CTPI_day', 'CTPI_evening', 'CTPI_night', 'CTPI_normalized']].describe().round(2))

print(f"\n最高时间贫困指数小区(TOP 10):")
display(aoi_ctpi.nlargest(10, 'CTPI_total')[
    ['name', 'community_type', 'CTPI_total', 'CTPI_normalized', 
     'CTPI_day', 'CTPI_evening', 'CTPI_night']
])

In [ ]:
# CTPI分级
def classify_ctpi_level(ctpi_normalized):
    """
    将CTPI标准化分数分为5个等级
    
    等级定义:
    - 0-20: 低贫困 (设施充足)
    - 20-40: 较低贫困
    - 40-60: 中等贫困
    - 60-80: 较高贫困
    - 80-100: 高贫困 (严重服务不足)
    """
    if ctpi_normalized <= 20:
        return '低贫困'
    elif ctpi_normalized <= 40:
        return '较低贫困'
    elif ctpi_normalized <= 60:
        return '中等贫困'
    elif ctpi_normalized <= 80:
        return '较高贫困'
    else:
        return '高贫困'


aoi_ctpi['poverty_level'] = aoi_ctpi['CTPI_normalized'].apply(classify_ctpi_level)

print("时间贫困等级分布:")
print(aoi_ctpi['poverty_level'].value_counts())

print("\n各贫困等级的CTPI均值:")
display(aoi_ctpi.groupby('poverty_level').agg({
    'CTPI_total': 'mean',
    'CTPI_normalized': 'mean',
    'community_id': 'count',
    'population': 'sum'
}).round(2))

---

## # 8. 服务盲区识别

基于可达性阈值和CTPI指数，识别服务盲区并生成优化优先级。

In [ ]:
# 定义关键设施盲区阈值
CRITICAL_FACILITIES = {
    '便利店': 500,      # 超过500m无便利店
    '药店': 800,       # 超过800m无药店
    '诊所': 800,       # 超过800m无诊所
    '地铁站': 1500,    # 超过1500m无地铁
    '公交站': 500,     # 超过500m无公交
}

# 设施类型英文到中文映射
EN2CN = {v: k for k, v in CATEGORY_NAMES_CN.items()}

def identify_service_gaps(df, critical_facilities, dist_prefix='dist_'):
    """
    识别服务盲区
    
    返回:
        添加盲区标记列的DataFrame
    """
    df = df.copy()
    
    df['critical_gap_count'] = 0
    df['critical_gap_types'] = ''
    
    for cn_name, threshold in critical_facilities.items():
        en_type = EN2CN.get(cn_name)
        if en_type:
            dist_col = f'{dist_prefix}{en_type}'
            if dist_col in df.columns:
                df[f'gap_{en_type}'] = df[dist_col] > threshold
                df.loc[df[f'gap_{en_type}'], 'critical_gap_count'] += 1
                df.loc[df[f'gap_{en_type}'], 'critical_gap_types'] += cn_name + '; '
    
    return df


aoi_gaps = identify_service_gaps(aoi_ctpi, CRITICAL_FACILITIES)

# 统计服务盲区
print("=" * 60)
print("【服务盲区识别结果】")
print("=" * 60)

gap_cols = [c for c in aoi_gaps.columns if c.startswith('gap_')]
for col in gap_cols:
    en_type = col.replace('gap_', '')
    cn_name = CATEGORY_NAMES_CN.get(en_type, en_type)
    gap_count = aoi_gaps[col].sum()
    gap_rate = gap_count / len(aoi_gaps) * 100
    threshold = CRITICAL_FACILITIES.get(cn_name, 'N/A')
    print(f"{cn_name}(>{threshold}m): {gap_count}个小区 ({gap_rate:.1f}%)")

print(f"\n多个关键设施盲区小区:")
display(aoi_gaps[aoi_gaps['critical_gap_count'] >= 2][[
    'name', 'community_type', 'critical_gap_count', 
    'critical_gap_types', 'CTPI_normalized', 'poverty_level'
]].sort_values('critical_gap_count', ascending=False))

In [ ]:
# 生成优化优先级排名
def generate_priority_ranking(df):
    """
    综合CTPI和盲区数量生成优化优先级
    
    优先级分数 = CTPI_normalized * 0.6 + critical_gap_count * 20 * 0.4
    """
    df = df.copy()
    df['priority_score'] = (
        df['CTPI_normalized'] * 0.6 + 
        df['critical_gap_count'] * 20 * 0.4
    )
    
    # 优先级分级
    def get_priority(score):
        if score >= 70:
            return 'P1-紧急'
        elif score >= 50:
            return 'P2-高'
        elif score >= 30:
            return 'P3-中'
        else:
            return 'P4-低'
    
    df['priority'] = df['priority_score'].apply(get_priority)
    df['priority_rank'] = df['priority_score'].rank(ascending=False).astype(int)
    
    return df.sort_values('priority_rank')


aoi_priority = generate_priority_ranking(aoi_gaps)

print("优化优先级排名(TOP 15):")
display(aoi_priority[['priority_rank', 'name', 'community_type', 
                        'CTPI_normalized', 'critical_gap_count', 
                        'priority', 'population']].head(15))

---

## # 9. ArcGIS兼容性导出

导出多种格式以适配ArcGIS Pro/QGIS等GIS软件。

In [ ]:
# ArcGIS字段映射表
print("=" * 60)
print("【ArcGIS字段映射表】")
print("=" * 60)

field_mapping = pd.DataFrame({
    '原始字段': [
        'community_id', 'name', 'type', 'community_type',
        'center_lng', 'center_lat', 'centroid_lng', 'centroid_lat',
        'area_km2', 'floor_count', 'built_year', 'population',
        'CTPI_total', 'CTPI_normalized', 'CTPI_day', 'CTPI_evening', 'CTPI_night',
        'poverty_level', 'priority', 'priority_rank', 'priority_score',
        'critical_gap_count', 'critical_gap_types',
    ] + [f'count_{ft}' for ft in facility_buffers.keys()] 
               + [f'dist_{ft}' for ft in facility_buffers.keys()],
    '中文说明': [
        '小区ID', '小区名称', '原始分类', '社区类型',
        '中心经度', '中心纬度', '质心经度', '质心纬度',
        '面积(km²)', '楼层数', '建成年份', '人口',
        'CTPI总分', 'CTPI标准化', '白天CTPI', '晚间CTPI', '深夜CTPI',
        '贫困等级', '优先级', '优先级排名', '优先级分数',
        '关键设施盲区数', '盲区设施列表',
    ] + [f'{CATEGORY_NAMES_CN.get(ft, ft)}数量' for ft in facility_buffers.keys()]
               + [f'{CATEGORY_NAMES_CN.get(ft, ft)}最近距离' for ft in facility_buffers.keys()],
    'ArcGIS建议': [
        'OBJECTID替代', '社区名称', '类型', '分类',
        'CenterX', 'CenterY', 'CentroidX', 'CentroidY',
        'Shape_Area(自动)', '楼层', 'Year', 'Pop',
        'CTPI_TOT', 'CTPI_NORM', 'CTPI_DAY', 'CTPI_EVE', 'CTPI_NGT',
        'Pov_Lvl', 'Priority', 'Rank', 'Score',
        'GapCnt', 'GapTypes',
    ] + [f'{ft[:10]}Cnt' for ft in facility_buffers.keys()]
               + [f'{ft[:10]}Dst' for ft in facility_buffers.keys()]
})

display(field_mapping)

In [ ]:
# 准备导出数据
export_df = aoi_priority.copy()

# 1. 导出GeoJSON
geojson_path = os.path.join(BASE_DIR, 'nanshan_community_ctpi.geojson')
export_gdf = gpd.GeoDataFrame(export_df, geometry='geometry')
export_gdf.to_file(geojson_path, driver='GeoJSON', encoding='utf-8')
print(f"✓ GeoJSON导出: {geojson_path}")

# 2. 导出Shapefile（处理字段名长度限制）
def shorten_field_names(gdf, max_length=10):
    """将长字段名截断到ArcGIS Shapefile限制的10字符以内"""
    gdf = gdf.copy()
    rename_map = {}
    for col in gdf.columns:
        if len(col) > max_length:
            # 取前8字符+序号
            new_name = col[:max_length]
            # 处理重名
            counter = 1
            base_name = new_name
            while new_name in rename_map.values():
                new_name = f"{base_name[:max_length-1]}{counter}"
                counter += 1
            rename_map[col] = new_name
    
    return gdf.rename(columns=rename_map), rename_map

shp_gdf, field_rename_map = shorten_field_names(export_gdf)
shp_path = os.path.join(BASE_DIR, 'nanshan_community_ctpi')
shp_gdf.to_file(shp_path, driver='ESRI Shapefile', encoding='utf-8')
print(f"✓ Shapefile导出: {shp_path}")

# 3. 导出CSV（含坐标列，供ArcGIS XY导入）
csv_df = export_df.drop(columns=['geometry', 'centroid', 'polygon'], errors='ignore')
csv_path = os.path.join(BASE_DIR, 'nanshan_community_ctpi.csv')
csv_df.to_csv(csv_path, index=False, encoding='utf-8-sig')
print(f"✓ CSV导出: {csv_path}")

# 4. 导出POI数据
poi_export = poi_cleaned.drop(columns=['geometry'], errors='ignore')
poi_csv_path = os.path.join(BASE_DIR, 'nanshan_poi_cleaned.csv')
poi_export.to_csv(poi_csv_path, index=False, encoding='utf-8-sig')
print(f"✓ POI CSV导出: {poi_csv_path}")

print(f"\n导出完成！共处理 {len(export_df)} 个小区，{len(poi_cleaned)} 个POI")

---

## # 10. 可视化输出

生成研究区专题地图和统计分析图表。

In [ ]:
# ============================================================================
# 10.1 设施分布地图 (Folium交互地图)
# ============================================================================

center_lat = (STUDY_AREA['lat_min'] + STUDY_AREA['lat_max']) / 2
center_lng = (STUDY_AREA['lng_min'] + STUDY_AREA['lng_max']) / 2

m = folium.Map(
    location=[center_lat, center_lng],
    zoom_start=13,
    tiles='CartoDB positron'
)

# 设施类型颜色映射
type_colors = {
    'convenience_store': 'green',
    'pharmacy': 'red',
    'hospital': 'darkred',
    'clinic': 'orange',
    'express_station': 'purple',
    'bank': 'blue',
    'atm': 'lightblue',
    'market': 'cadetblue',
    'school': 'darkgreen',
    'kindergarten': 'lightgreen',
    'bus_station': 'gray',
    'subway': 'black',
    'park': 'forestgreen',
    'restaurant': 'beige',
    'cafe': 'pink',
    'gym': 'darkblue',
    'playground': 'lightred'
}

# 添加POI点
feature_group_by_type = {}
for ft in poi_cleaned['standard_type'].unique():
    fg = folium.FeatureGroup(name=CATEGORY_NAMES_CN.get(ft, ft))
    feature_group_by_type[ft] = fg
    
    subset = poi_cleaned[poi_cleaned['standard_type'] == ft]
    for _, row in subset.iterrows():
        folium.CircleMarker(
            location=[row['lat'], row['lng']],
            radius=4,
            color=type_colors.get(ft, 'gray'),
            fill=True,
            fillColor=type_colors.get(ft, 'gray'),
            fillOpacity=0.7,
            popup=f"{CATEGORY_NAMES_CN.get(ft, ft)}<br>{row['name']}"
        ).add_to(fg)
    fg.add_to(m)

# 添加小区AOI（按贫困等级着色）
poverty_colors = {
    '低贫困': '#2ecc71',
    '较低贫困': '#f1c40f',
    '中等贫困': '#e67e22',
    '较高贫困': '#e74c3c',
    '高贫困': '#8e44ad'
}

fg_community = folium.FeatureGroup(name='小区贫困等级')
for _, row in aoi_priority.iterrows():
    color = poverty_colors.get(row['poverty_level'], 'gray')
    folium.GeoJson(
        row['geometry'].__geo_interface__,
        style_function=lambda x, c=color: {
            'fillColor': c,
            'color': 'black',
            'weight': 1,
            'fillOpacity': 0.4
        },
        popup=folium.Popup(
            f"<b>{row['name']}</b><br>"
            f"类型: {row['community_type']}<br>"
            f"CTPI: {row['CTPI_normalized']:.1f}<br>"
            f"贫困等级: {row['poverty_level']}<br>"
            f"优先级: {row['priority']}<br>"
            f"人口: {row['population']}人",
            max_width=250
        )
    ).add_to(fg_community)
fg_community.add_to(m)

# 添加图层控制
folium.LayerControl().add_to(m)

folium_map_path = os.path.join(BASE_DIR, 'nanshan_accessibility_map.html')
m.save(folium_map_path)
print(f"✓ Folium交互地图已保存: {folium_map_path}")

display(m)

In [ ]:
# ============================================================================
# 10.2 CTPI热力图 (matplotlib)
# ============================================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# 2.1 各小区CTPI分布散点图
ax1 = axes[0, 0]
colors_by_level = {
    '低贫困': '#2ecc71', '较低贫困': '#f1c40f',
    '中等贫困': '#e67e22', '较高贫困': '#e74c3c', '高贫困': '#8e44ad'
}
for level in ['低贫困', '较低贫困', '中等贫困', '较高贫困', '高贫困']:
    subset = aoi_priority[aoi_priority['poverty_level'] == level]
    if len(subset) > 0:
        ax1.scatter(subset['centroid_lng'], subset['centroid_lat'],
                   c=colors_by_level[level], s=subset['CTPI_normalized']*3+30,
                   alpha=0.7, label=f'{level}({len(subset)})', edgecolors='white')
ax1.set_title('小区CTPI空间分布', fontsize=14, fontweight='bold')
ax1.set_xlabel('经度')
ax1.set_ylabel('纬度')
ax1.legend(title='贫困等级', loc='upper left', fontsize=8)
ax1.grid(True, alpha=0.3)

# 2.2 CTPI直方图
ax2 = axes[0, 1]
bins = np.linspace(0, 100, 21)
colors_hist = [colors_by_level.get(classify_ctpi_level(b+5), 'gray') for b in bins[:-1]]
n, bins_out, patches = ax2.hist(aoi_priority['CTPI_normalized'], bins=bins, 
                                edgecolor='white', alpha=0.8)
for i, patch in enumerate(patches):
    patch.set_facecolor(list(colors_by_level.values())[min(i//4, 4)])
ax2.set_title('CTPI标准化分数分布', fontsize=14, fontweight='bold')
ax2.set_xlabel('CTPI标准化分数 (0-100)')
ax2.set_ylabel('小区数量')
ax2.axvline(aoi_priority['CTPI_normalized'].mean(), color='red', 
           linestyle='--', linewidth=2, label=f"均值: {aoi_priority['CTPI_normalized'].mean():.1f}")
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

# 2.3 各时段CTPI对比
ax3 = axes[1, 0]
period_names = ['白天\n(06-18时)', '晚间\n(18-23时)', '深夜\n(23-06时)']
period_cols = ['CTPI_day', 'CTPI_evening', 'CTPI_night']
period_colors = ['#f39c12', '#e74c3c', '#3498db']

bp_data = [aoi_priority[col] for col in period_cols]
bp = ax3.boxplot(bp_data, labels=period_names, patch_artist=True,
                  medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp['boxes'], period_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax3.set_title('各时段CTPI分布对比', fontsize=14, fontweight='bold')
ax3.set_ylabel('CTPI分数')
ax3.grid(True, alpha=0.3, axis='y')

# 2.4 社区类型CTPI对比
ax4 = axes[1, 1]
type_ctpi = aoi_priority.groupby('community_type')['CTPI_normalized'].mean().sort_values(ascending=False)
bars = ax4.bar(type_ctpi.index, type_ctpi.values, 
               color=['#8e44ad', '#e74c3c', '#e67e22', '#f1c40f'], alpha=0.8,
               edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, type_ctpi.values):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.1f}', ha='center', fontsize=11, fontweight='bold')
ax4.set_title('各社区类型平均CTPI', fontsize=14, fontweight='bold')
ax4.set_ylabel('平均CTPI标准化分数')
ax4.set_xlabel('社区类型')
ax4.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'ctpi_analysis_charts.png'), dpi=150, bbox_inches='tight')
plt.show()
print("图表已保存: ctpi_analysis_charts.png")

In [ ]:
# ============================================================================
# 10.3 设施覆盖热力图
# ============================================================================

fig, ax = plt.subplots(figsize=(14, 10))

# 按人口加权绘制气泡图
for idx, row in aoi_priority.iterrows():
    total_facilities = sum(row[f'count_{ft}'] for ft in facility_buffers.keys())
    bubble_size = (row['population'] / 100) + 10  # 人口越多气泡越大
    bubble_color = poverty_colors.get(row['poverty_level'], 'gray')
    
    ax.scatter(row['centroid_lng'], row['centroid_lat'],
              s=bubble_size*5, c=bubble_color, alpha=0.6,
              edgecolors='black', linewidths=0.5)

# 添加色标
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=v, label=k, edgecolor='black') 
                    for k, v in poverty_colors.items()]
ax.legend(handles=legend_elements, title='贫困等级', loc='upper left', fontsize=9)

ax.set_title('南山区小区时间贫困分布图\n(气泡大小=人口, 颜色=贫困等级)', fontsize=16, fontweight='bold')
ax.set_xlabel('经度', fontsize=12)
ax.set_ylabel('纬度', fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'community_poverty_map.png'), dpi=150, bbox_inches='tight')
plt.show()
print("图表已保存: community_poverty_map.png")

In [ ]:
# ============================================================================
# 10.4 关键设施盲区分布
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 4.1 各设施盲区数量
ax1 = axes[0]
gap_stats = []
for col in gap_cols:
    en_type = col.replace('gap_', '')
    cn_name = CATEGORY_NAMES_CN.get(en_type, en_type)
    gap_count = aoi_gaps[col].sum()
    gap_stats.append({'设施': cn_name, '盲区数量': gap_count})

gap_df = pd.DataFrame(gap_stats).sort_values('盲区数量', ascending=True)
ax1.barh(gap_df['设施'], gap_df['盲区数量'], color='salmon', edgecolor='darkred')
for i, (idx, row) in enumerate(gap_df.iterrows()):
    ax1.text(row['盲区数量'] + 0.3, i, f"{row['盲区数量']}个", va='center', fontsize=10)
ax1.set_title('关键设施服务盲区统计', fontsize=14, fontweight='bold')
ax1.set_xlabel('无该设施的小区数量')
ax1.grid(True, alpha=0.3, axis='x')

# 4.2 设施密度分布（按类型）
ax2 = axes[1]
count_stats = []
for ft in facility_buffers.keys():
    col = f'count_{ft}'
    cn_name = CATEGORY_NAMES_CN.get(ft, ft)
    total = aoi_priority[col].sum()
    avg = aoi_priority[col].mean()
    count_stats.append({'设施': cn_name, '总数': total, '户均': avg})

count_df = pd.DataFrame(count_stats).sort_values('户均', ascending=True)
ax2.barh(count_df['设施'], count_df['户均'], color='steelblue', edgecolor='navy', alpha=0.8)
ax2.set_title('各类型设施户均覆盖数量', fontsize=14, fontweight='bold')
ax2.set_xlabel('户均设施数量')
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'facility_gap_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()
print("图表已保存: facility_gap_analysis.png")

---

## # 11. 数据质量报告

生成完整的数据质量评估报告，记录数据清洗和分析过程中的关键指标。

In [ ]:
# ============================================================================
# 生成数据质量报告
# ============================================================================

def generate_data_quality_report(poi_df, aoi_df, crs_report, 
                                  poi_cleaned_df, aoi_ctpi_df,
                                  base_dir=BASE_DIR):
    """
    生成完整数据质量报告
    """
    report = {}
    
    # 1. POI数据质量
    report['poi'] = {
        '原始记录数': len(poi_df),
        '清洗后记录数': len(poi_cleaned_df),
        '坐标系统一': {
            '原始GCJ02': crs_report['gcj02_count'],
            '原始WGS84': crs_report['wgs84_count'],
            '已转换': crs_report['transformed_count']
        },
        '缺失值统计': poi_cleaned_df.isnull().sum().to_dict(),
        '坐标异常': int((~poi_cleaned_df['coord_valid']).sum()),
        '离群点': int(poi_cleaned_df['is_outlier'].sum()),
        '24h设施': int(poi_cleaned_df['is_24h'].sum()),
        '设施类型数': poi_cleaned_df['standard_type'].nunique(),
        '类型分布': poi_cleaned_df['standard_type'].value_counts().to_dict()
    }
    
    # 2. AOI数据质量
    report['aoi'] = {
        '小区总数': len(aoi_df),
        '有效几何': int(aoi_df['is_valid'].sum()),
        '无效几何': int((~aoi_df['is_valid']).sum()),
        '社区类型': aoi_df['community_type'].value_counts().to_dict(),
        '总人口': int(aoi_df['population'].sum()),
        '平均面积(km²)': round(aoi_df['area_km2'].mean(), 4),
        'CTPI统计': {
            '均值': round(aoi_ctpi_df['CTPI_normalized'].mean(), 2),
            '中位数': round(aoi_ctpi_df['CTPI_normalized'].median(), 2),
            '最大值': round(aoi_ctpi_df['CTPI_normalized'].max(), 2),
            '最小值': round(aoi_ctpi_df['CTPI_normalized'].min(), 2)
        },
        '贫困等级分布': aoi_ctpi_df['poverty_level'].value_counts().to_dict()
    }
    
    # 3. 导出文件列表
    import glob
    files = glob.glob(os.path.join(base_dir, '*'))
    report['export_files'] = [os.path.basename(f) for f in files 
                              if os.path.isfile(f)]
    
    return report


quality_report = generate_data_quality_report(
    poi_df_transformed, aoi_clipped, crs_report,
    poi_cleaned, aoi_ctpi
)

print("=" * 70)
print("【数据质量报告】")
print("=" * 70)

print("\n一、POI数据质量")
print(f"  原始记录: {quality_report['poi']['原始记录数']}条")
print(f"  清洗后记录: {quality_report['poi']['清洗后记录数']}条")
print(f"  坐标系统一: GCJ02→WGS84 转换 {quality_report['poi']['坐标系统一']['已转换']}条")
print(f"  坐标异常: {quality_report['poi']['坐标异常']}条")
print(f"  离群点检测: {quality_report['poi']['离群点']}个")
print(f"  24h设施: {quality_report['poi']['24h设施']}个")
print(f"  标准设施类型: {quality_report['poi']['设施类型数']}种")

print("\n二、小区AOI数据质量")
print(f"  小区总数: {quality_report['aoi']['小区总数']}个")
print(f"  有效几何: {quality_report['aoi']['有效几何']}个")
print(f"  无效几何: {quality_report['aoi']['无效几何']}个")
print(f"  覆盖总人口: {quality_report['aoi']['总人口']}人")
print(f"  平均小区面积: {quality_report['aoi']['平均面积(km²)']}km²")

print("\n三、CTPI分析结果")
ctpi_stats = quality_report['aoi']['CTPI统计']
print(f"  平均CTPI: {ctpi_stats['均值']}")
print(f"  中位数CTPI: {ctpi_stats['中位数']}")
print(f"  最高CTPI: {ctpi_stats['最大值']}")
print(f"  最低CTPI: {ctpi_stats['最小值']}")

print("\n四、贫困等级分布")
for level, count in quality_report['aoi']['贫困等级分布'].items():
    pct = count / quality_report['aoi']['小区总数'] * 100
    print(f"  {level}: {count}个小区 ({pct:.1f}%)")

print("\n五、导出文件")
for fname in sorted(quality_report['export_files']):
    print(f"  - {fname}")

# 保存报告为JSON
report_path = os.path.join(BASE_DIR, 'data_quality_report.json')
with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(quality_report, f, ensure_ascii=False, indent=2, default=str)
print(f"\n报告已保存: {report_path}")

In [ ]:
# 最终数据摘要表格
print("\n" + "=" * 70)
print("【分析结果摘要】")
print("=" * 70)

summary_df = pd.DataFrame({
    '指标': [
        '研究区总面积',
        'POI设施总数',
        '标准设施类型数',
        '小区总数',
        '覆盖总人口',
        '平均CTPI',
        '高贫困小区数',
        'P1紧急优先级',
        'P2高优先级',
        '关键设施盲区'
    ],
    '数值': [
        '~185 km²',
        f'{len(poi_cleaned)} 个',
        f'{poi_cleaned["standard_type"].nunique()} 种',
        f'{len(aoi_priority)} 个',
        f'{aoi_priority["population"].sum()} 人',
        f'{aoi_priority["CTPI_normalized"].mean():.1f}',
        f'{(aoi_priority["poverty_level"]=="高贫困").sum()} 个',
        f'{(aoi_priority["priority"]=="P1-紧急").sum()} 个',
        f'{(aoi_priority["priority"]=="P2-高").sum()} 个',
        f'{aoi_gaps["critical_gap_count"].sum()} 处'
    ]
})

display(summary_df)

---

**分析完成！**  

本Notebook完成了以下工作:

1. **数据导入** — POI + 小区AOI 模拟数据生成
2. **坐标系统一** — GCJ-02 → WGS-84 转换
3. **研究区裁剪** — 南山区 bounding box 过滤
4. **POI清洗** — 类型标准化、营业时间、去重、离群点检测
5. **AOI处理** — 质心计算、几何验证、社区分类
6. **空间关联** — 500m步行缓冲区分析、空间连接、最近设施距离
7. **CTPI计算** — 三时段权重矩阵、时间贫困指数计算
8. **盲区识别** — 关键设施服务盲区 + 优化优先级
9. **ArcGIS导出** — GeoJSON、Shapefile、CSV 多格式兼容
10. **可视化** — Folium交互地图 + 统计分析图表
11. **质量报告** — JSON格式完整数据质量评估

下一步建议:
- 接入真实POI数据(高德/百度API)
- 接入真实小区边界数据(规划局/测绘院)
- 引入实时交通数据优化步行可达性
- 与人口普查数据叠加分析